# On-resonance timestream calibration

In [ ]:
import os
import numpy as np
from citkid.pipeline.dataset import DataSet
from citkid.pipeline.analysis import AnalysisRunner
from citkid.pipeline.interactive import run_ts_analysis

zpath = os.path.normpath(r'path/to/ts_analysis.zarr')
cpath = 'custom_steps_ts.py'

DS = DataSet(
    custom_path    = cpath,
    zarr_path      = zpath,
    cal_yaml_path  = 'ts',
    zarr_mode      = 'a', 
)
AR = AnalysisRunner(
    DS,
    analysis_yaml_path = 'ts',
    custom_path        = cpath,
)

In [ ]:
data_idxs_to_analyze = np.where(DS.res_idxs[:] >= 0)[0] # res_idxs >= 0 for on-res
run_ts_analysis(
    AR,
    start_idx  = 0,
    data_idxs  = data_idxs_to_analyze,
    ui_scale   = 0.7,
    plot_scale = 0.5,
)

# Off-resonance timestream calibration

In [ ]:
import os
import numpy as np
from citkid.pipeline.dataset import DataSet
from citkid.pipeline.analysis import AnalysisRunner
from citkid.pipeline.interactive import run_gain_only_analysis

zpath = os.path.normpath(r'path/to/ts_analysis.zarr')
cpath = 'custom_steps_ts.py'

DS = DataSet(
    custom_path    = cpath,
    zarr_path      = zpath,
    cal_yaml_path  = 'ts',
    zarr_mode      = 'a',
)
AR = AnalysisRunner(
    DS,
    analysis_yaml_path = 'ts_offres',
    custom_path        = cpath,
)

In [ ]:
data_idxs_to_analyze = np.where(DS.res_idxs[:] < 0)[0] # res_idx < 0 for off-res
run_gain_only_analysis(
    AR,
    start_idx  = 0,
    data_idxs  = data_idxs_to_analyze,
    ui_scale   = 0.7,
    plot_scale = 0.5,
)

# IQ fitting

In [ ]:
import os
import numpy as np
from citkid.pipeline.dataset import DataSet
from citkid.pipeline.analysis import AnalysisRunner
from citkid.pipeline.interactive import run_iq_analysis

zpath = os.path.normpath(r'path/to/iq_analysis.zarr')
cpath = 'custom_steps_iq.py'

DS = DataSet(
    custom_path    = cpath,
    zarr_path      = zpath,
    cal_yaml_path  = 'iq',
    zarr_mode      = 'a',
)
AR = AnalysisRunner(
    DS,
    analysis_yaml_path = 'iq',
    custom_path        = cpath,
)

In [ ]:
data_idxs_to_analyze = np.where(DS.res_idxs[:] >= 0)[0]
run_iq_analysis(
    AR,
    start_idx  = 0,
    data_idxs  = data_idxs_to_analyze,
    ui_scale   = 0.7,
    plot_scale = 0.5,
)

# Implementation with flagging

In [ ]:
import os
import numpy as np
from citkid.pipeline.dataset import DataSet
from citkid.pipeline.analysis import AnalysisRunner
from citkid.pipeline.interactive import run_ts_analysis

zpath = os.path.normpath(r'path/to/ts_analysis.zarr')
cpath = 'custom_steps_ts.py'

DS = DataSet(
    custom_path    = cpath,
    zarr_path      = zpath,
    cal_yaml_path  = 'ts',
    zarr_mode      = 'a', 
)
AR = AnalysisRunner(
    DS,
    analysis_yaml_path = 'ts',
    custom_path        = cpath,
)

In [ ]:
# Execute path for all 
on_res_idxs = np.where(DS.res_idxs[:] >= 0)[0] 
AR.execute_path(data_idx = on_res_idxs, verbose=True)

In [ ]:
# Create flags 
# Example -> 5 Hz sxx exceeds 1e-15 
# Alternatively, sxx could be added as a calibration step, and the flagging 
# would simplify to 
#       data_idxs_to_analyze = on_res_idxs[DS.sxx[on_res_idxs] < 1e-15]
from citkid.signal.psd import get_psd 
sxx5 = np.empty(DS.nrows, dtype = np.float64) 
for data_idx in range(DS.nrows):
    if data_idx in on_res_idxs:
        f, sxx = get_psd(DS.x[data_idx], DS.dt, get_frequencies = True)
        sxx5[data_idx] = np.mean(sxx[np.abs(f - 5) < 2])
    else:
        sxx5[data_idx] = np.nan
data_idxs_to_analyze = np.where(np.isfinite(sxx5) & (sxx5 > 1e-15))[0]

In [ ]:
run_gain_only_analysis(
    AR,
    start_idx  = 0,
    data_idxs  = data_idxs_to_analyze,
    ui_scale   = 0.7,
    plot_scale = 0.5,
)